# Hallucinations & Grounding + RAG Evaluation Basics
### Practice Notebook (Deliverable Work Session)

**Assumed pre-installed libraries:** `numpy`, `scikit-learn`,
`sentence-transformers`, `chromadb`

This notebook is the capstone work session for the week: you'll assemble the
retrieval pipeline from Days 2–4 into a small end-to-end system, then build
simple metrics to evaluate whether its answers are actually grounded in the
retrieved context.


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from chromadb.utils import embedding_functions
import numpy as np
import chromadb

## 1. Why LLMs hallucinate

A few root causes worth covering explicitly in class:

1. **Next-token prediction, not fact lookup** — the model is optimized to
   produce plausible-sounding continuations, not to check a claim against
   a source of truth. Fluency and correctness are optimized differently.
2. **Knowledge gaps** — questions about things outside training data (recent
   events, private data, long-tail facts) leave the model "guessing" from
   loosely related patterns.
3. **Prompt ambiguity / underspecification** — a vague question invites the
   model to fill gaps with invented specifics.
4. **Compounding errors in reasoning chains** — a small early error (e.g., a
   wrong intermediate number) can snowball into a confidently wrong final
   answer, especially in longer chain-of-thought generations.
5. **Sycophancy / pattern completion** — the model may complete a pattern the
   user's phrasing implies is true, even when it isn't (e.g., a leading
   question that assumes a false premise).

RAG doesn't eliminate hallucination — it reduces one specific class of it
(missing/out-of-date knowledge) by handing the model a source to work from.
But the model can still ignore or misread that source. That's what we'll
measure below.


## 2. Grounding techniques to reduce hallucination

- **Retrieval + explicit instruction** — "answer ONLY using the context
  below; say 'I don't know' if it's not there" (you already did this on
  Day 1's `augment_prompt`).
- **Citations** — require the model to tag each claim with the source chunk
  it came from; makes ungrounded claims easy to spot (no citation = suspect).
- **Confidence/abstention thresholds** — if retrieval similarity scores are
  all low (nothing relevant was found), have the system refuse to answer
  rather than forcing a guess.
- **Answer verification pass** — a second LLM call (or a lighter heuristic
  check) that checks whether the generated answer is actually supported by
  the retrieved context, before returning it to the user.
- **Structured output constraints** — asking for answers in a format that's
  easy to verify programmatically (e.g., a JSON object with an `evidence`
  field) rather than free-flowing prose.

Let's build a simple version of the last two: an abstention threshold and a
lightweight groundedness check.


## 3. Assemble the mini end-to-end RAG pipeline

We'll reuse the same building blocks from earlier in the week: chunking
(Day 2), embeddings + semantic search (Day 3), and a vector store (Day 4).


In [4]:

chroma_client = chromadb.Client()
local_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")
collection = chroma_client.create_collection(
    name="rag_eval_practice",
    embedding_function=local_ef
)

knowledge_base = [
    {"id": "doc1", "text": "Our Q3 refund policy allows exceptions for orders "
                            "delayed more than 10 business days, approved by a "
                            "supervisor. Order #48213 was delayed 14 days and "
                            "was granted a full refund on 2025-08-02."},
    {"id": "doc2", "text": "Standard refund policy: refunds are issued within "
                            "5-7 business days of a return being received."},
    {"id": "doc3", "text": "Our shipping partners include BlueDart, Delhivery, "
                            "and DTDC for domestic orders across India."},
    {"id": "doc4", "text": "Customer support is available 9am-9pm IST, "
                            "Monday through Saturday, via chat and email."},
]

collection.add(
    documents=[d["text"] for d in knowledge_base],
    ids=[d["id"] for d in knowledge_base],
)
print("Indexed", collection.count(), "documents")


Indexed 4 documents


In [ ]:
def retrieve(query: str, k: int = 2):
    results = collection.query(query_texts=[query], n_results=k)
    docs = results["documents"][0]
    distances = results["distances"][0]
    # Chroma's default distance is squared L2. For unit-normalized embeddings,
    # squared L2 distance d relates to cosine similarity as: cos_sim = 1 - d/2.
    # We surface BOTH numbers below -- do not trust a single "similarity-like"
    # score without checking what it actually implies.
    scores = [1 / (1 + d) for d in distances]
    implied_cosine = [1 - d / 2 for d in distances]
    return list(zip(docs, scores, implied_cosine))

def generate_stub(query: str, retrieved: list, min_score: float = 0.6) -> dict:
    """Stub generator with an ABSTENTION THRESHOLD: if nothing retrieved
    clears min_score (on the implied-cosine scale), refuse to answer instead
    of guessing.
    IMPORTANT: 0.6 is NOT a universal safe number -- it was picked after
    inspecting this specific corpus and embedding model's score spread (see
    Exercise 5.1). A different corpus, embedding model, or query set could
    shift "unrelated" scores above or below this line. Never hard-code a
    threshold without checking it against real relevant/irrelevant examples
    for your actual data.
    Replace the body with a real LLM call in production:
        # TODO: replace with a real LLM call, still passing `context`
    """
    usable = [(doc, cos) for doc, score, cos in retrieved if cos >= min_score]
    if not usable:
        return {"answer": "I don't have enough information to answer that.",
                "grounded_in": [], "abstained": True}
    context = " ".join(doc for doc, _ in usable)
    return {"answer": f"(stub) Based on available context: {context[:200]}...",
            "grounded_in": [doc for doc, _ in usable], "abstained": False}

for q in ["What's the refund policy for order #48213?",
          "What is the capital of France?"]:
    retrieved = retrieve(q)
    print("Query:", q)
    for doc, score, cos in retrieved:
        print(f"   score={score:.3f}  implied_cosine={cos:.3f}  {doc[:60]}...")
    result = generate_stub(q, retrieved) # Remove generate_stub and call the real llm call
    print("   ->", result["answer"][:120], "\n")


Query: What's the refund policy for order #48213?
   score=0.756  implied_cosine=0.838  Our Q3 refund policy allows exceptions for orders delayed mo...
   score=0.681  implied_cosine=0.765  Standard refund policy: refunds are issued within 5-7 busine...
   -> (stub) Based on available context: Our Q3 refund policy allows exceptions for orders delayed more than 10 business days, 

Query: What is the capital of France?
   score=0.512  implied_cosine=0.523  Our shipping partners include BlueDart, Delhivery, and DTDC ...
   score=0.491  implied_cosine=0.483  Standard refund policy: refunds are issued within 5-7 busine...
   -> I don't have enough information to answer that. 



**Exercise 5.1 (revised -- read before running):** Look at the `implied_cosine`
column printed above, not just `score`. Unrelated sentence-embedding
similarity commonly sits around 0.4-0.6 for models like MiniLM (a known
effect called *anisotropy* -- embeddings cluster in a narrow cone of the
vector space rather than spreading out toward 0 for "unrelated" text). This
means a naive threshold like the original `min_score=0.35` on the raw
`1/(1+d)` score would **not** reliably abstain on the "capital of France"
query -- it could easily clear that bar even though nothing in the
knowledge base is relevant.

Do the following:
1. Print `implied_cosine` for both queries above. Does the refund question
   score meaningfully higher than the capital-of-France question, or are
   they closer together than you'd expect?
2. Try `min_score` values of `0.4`, `0.6`, and `0.8` in `generate_stub`.
   For each, does the capital-of-France query abstain? Does the *legitimate*
   refund question still get answered?
3. Based on what you find, is there a single threshold that cleanly
   separates "relevant" from "irrelevant" for this tiny 4-document corpus?
   If not, what does that tell you about relying on a fixed embedding-
   similarity cutoff for real-world abstention, and what would you use
   instead? (Hint: re-read Day 5's grounding-techniques discussion on
   LLM-as-judge / verification passes -- this is exactly the gap they exist
   to cover.)

**Then:** set `min_score = 0.0` and re-run. What does the stub generator do
now, and why is that dangerous in a real system?


## 4. RAG evaluation basics: relevance, faithfulness, groundedness

Three commonly used metrics for evaluating a RAG system (this is the
simplified, from-scratch version of what libraries like `ragas` compute):

- **Context relevance** — how relevant is each *retrieved chunk* to the
  question? (Are we retrieving the right thing?)
- **Faithfulness** — does the *generated answer* only contain claims that
  are actually supported by the retrieved context? (Is the model
  hallucinating on top of good retrieval?)
- **Groundedness** — similar to faithfulness; often used to mean "can every
  claim in the answer be traced back to a specific source passage."

We'll approximate all three with embedding-similarity heuristics, since we
don't have a real LLM judge wired in here.


In [7]:
def context_relevance(query: str, retrieved_docs: list) -> float:
    """Average cosine similarity between the query and each retrieved
    chunk. Higher = retrieval found genuinely relevant material.
    """
    query_emb = model.encode([query])
    doc_embs = model.encode(retrieved_docs)
    sims = cosine_similarity(query_emb, doc_embs)[0]
    return float(np.mean(sims))

def faithfulness(answer: str, retrieved_docs: list) -> float:
    """Approximate faithfulness as the max cosine similarity between the
    answer and any single retrieved chunk -- a real implementation would
    split the answer into individual claims and check each one separately
    against the context (e.g., with an LLM-as-judge or an NLI model).
    """
    if not retrieved_docs:
        return 0.0
    answer_emb = model.encode([answer])
    doc_embs = model.encode(retrieved_docs)
    sims = cosine_similarity(answer_emb, doc_embs)[0]
    return float(np.max(sims))

query = "What's the refund policy for order #48213?"
retrieved = retrieve(query)
retrieved_docs = [doc for doc, _score, _cos in retrieved]
result = generate_stub(query, retrieved)

print("Context relevance:", round(context_relevance(query, retrieved_docs), 3))
print("Faithfulness:      ", round(faithfulness(result["answer"], retrieved_docs), 3))


Context relevance: 0.604
Faithfulness:       0.957


In [8]:
retrieved

[('Our Q3 refund policy allows exceptions for orders delayed more than 10 business days, approved by a supervisor. Order #48213 was delayed 14 days and was granted a full refund on 2025-08-02.',
  0.7557103322159306,
  0.8383708298206329),
 ('Standard refund policy: refunds are issued within 5-7 business days of a return being received.',
  0.6805657080428956,
  0.7653170824050903)]

Why embedding similarity falls short here, concretely:

**Topical similarity ≠ factual support**.
A fabricated claim ("the refund was for ₹50,000") can score high in cosine similarity against the real refund-policy chunk, because it's about the same topic, uses the same vocabulary, and mentions the same entity — even though the specific number is wrong. Cosine similarity has no concept of "this specific fact is or isn't stated in this text"; it only measures overall semantic resemblance.

**NLI models are trained specifically for this**: given a "premise" (the retrieved context) and a "hypothesis" (the claim), they output entailment / contradiction / neutral — a much sharper signal than "how similar are these two vectors."

**LLM-as-judge goes further**: you can literally prompt a model with "Given this context, is the following claim fully supported, partially supported, or not supported? Quote the exact text that supports or contradicts it." This catches contradictions and unsupported specifics that embedding similarity structurally cannot detect, at the cost of an extra LLM call per claim (latency and money).

In [9]:
def groundedness_report(answer: str, retrieved_docs: list, threshold: float = 0.5) -> dict:
    """Split the answer into rough sentence-level claims and check each
    one against the best-matching retrieved chunk. Returns the fraction of
    claims that clear the similarity threshold ('grounded') plus the
    per-claim detail, so you can see exactly which parts of an answer are
    (or aren't) supported.
    """
    claims = [c.strip() for c in answer.split(".") if c.strip()]
    if not claims or not retrieved_docs:
        return {"grounded_fraction": 0.0, "details": []}

    claim_embs = model.encode(claims)
    doc_embs = model.encode(retrieved_docs)
    sims = cosine_similarity(claim_embs, doc_embs)

    details = []
    grounded_count = 0
    for claim, row in zip(claims, sims):
        best_score = float(np.max(row))
        is_grounded = best_score >= threshold
        grounded_count += is_grounded
        details.append({"claim": claim, "best_score": round(best_score, 3),
                         "grounded": is_grounded})

    return {"grounded_fraction": grounded_count / len(claims), "details": details}

report = groundedness_report(
    "Order 48213 got a full refund because it was delayed. "
    "The moon landing happened in 1969.",   # deliberately inject an unrelated claim
    retrieved_docs,
)
print("Grounded fraction:", round(report["grounded_fraction"], 3))
for d in report["details"]:
    print(f"  [{'OK' if d['grounded'] else 'UNGROUNDED'}] "
          f"(score={d['best_score']})  {d['claim']}")


Grounded fraction: 0.5
  [OK] (score=0.638)  Order 48213 got a full refund because it was delayed
  [UNGROUNDED] (score=0.0)  The moon landing happened in 1969


Notice the injected, unrelated claim about the moon landing scores low and
gets flagged as ungrounded — this is the core mechanism behind automated
hallucination detection in RAG evaluation: **check the generated answer's
claims against the retrieved context, not against general plausibility.**

**Exercise 5.2:** write your own `generate_stub`-style answer that mixes one
grounded claim with one fabricated (but plausible-sounding) claim about the
same domain (e.g., invent a fake refund amount). Run it through
`groundedness_report` — does it catch the fabrication? If not, why might a
simple cosine-similarity check fail here? (Hint: a fabricated claim can still
be *topically* similar to the context even if factually wrong — this is a
real limitation of embedding-based faithfulness checks, and part of why
production systems often use an LLM-as-judge instead.)


## 5. Deliverable: document retrieval experiments + RAG evaluation notebook

For this week's deliverable, extend the pipeline above into a small
experiment. Suggested structure:

1. **Build a knowledge base** of 8–12 short documents on a topic of your
   choice (reuse your Day 2 chunking exercise output, or write new ones).
2. **Write 5 test questions**, including:
   - 2 questions clearly answerable from the knowledge base
   - 1 question that's a paraphrase (tests semantic vs. keyword retrieval)
   - 1 question with **no** good answer in the knowledge base (tests
     abstention)
   - 1 "trick" question with a false premise (tests whether the model
     corrects it or plays along)
3. **Run each question** through `retrieve()` and `generate_stub()`, and
   record: the retrieved chunks, the (stub) answer, and whether abstention
   triggered correctly.
4. **Score each answer** with `context_relevance`, `faithfulness`, and
   `groundedness_report`, and present the results in a small table.
5. **Write a 4-6 sentence summary**: where did the pipeline do well? Where
   did it fail, and which stage (chunking, retrieval, or generation) was
   responsible for each failure?

This ties together every concept from the week: chunking → embeddings →
vector storage → retrieval → grounding → evaluation, as one connected
pipeline rather than five separate topics.
